In [17]:
from ccka.models.kernel import KernelModel, HardwareKernelRunner, HardwareKernelModel
from ccka.circuits.angleEmbeddingKernel import QuackEmbeddingQiskitCircuit
import pennylane as qml
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
import os
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [18]:
def _make_circular_data(num_sectors):
    """Generate datapoints arranged in an even circle."""
    center_indices = np.array(range(0, num_sectors))
    sector_angle = 2 * np.pi / num_sectors
    angles = (center_indices + 0.5) * sector_angle
    x = 0.7 * np.cos(angles)
    y = 0.7 * np.sin(angles)
    labels = 2 * np.remainder(np.floor_divide(angles, sector_angle), 2) - 1

    return x, y, labels


def make_double_cake_data(num_sectors):
    x1, y1, labels1 = _make_circular_data(num_sectors)
    x2, y2, labels2 = _make_circular_data(num_sectors)

    # x and y coordinates of the datapoints
    x = np.hstack([x1, 0.5 * x2])
    y = np.hstack([y1, 0.5 * y2])

    # Canonical form of dataset
    X = np.vstack([x, y]).T

    labels = np.hstack([labels1, -1 * labels2])

    # Canonical form of labels
    Y = labels.astype(int)

    return X, Y

X, y = make_double_cake_data(num_sectors=4)
X.shape, y.shape

((8, 2), (8,))

In [19]:
from qiskit_ibm_runtime import QiskitRuntimeService

# First time only — saves credentials to disk
QiskitRuntimeService.save_account(
    channel="ibm_quantum_platform",
    token= 'x1rF9ZkNHw10h3GaSBdm6PJ5CpyX8VkrNGYGOURdjbpv', #'TzFqvDVsCLogQCHMC8uMFVDqoXVesYpXef9i-48OCade',
    overwrite=True,
)

# Load saved account
service = QiskitRuntimeService()

# List available backends and pick the least busy with enough qubits
backend = service.least_busy(
   operational=True,
    simulator=False,
    min_num_qubits=2,          # match your num_qubits
)

print(f"Using backend: {backend.name}")

# ── Circuit and runner ─────────────────────────────────────────────────
num_qubits = 2
reps       = 1

circuit = QuackEmbeddingQiskitCircuit(num_qubits=num_qubits, reps=reps, reupload=True)
runner  = HardwareKernelRunner(
    circuit=circuit,
    backend=backend,
    shots=1024,
    batch_size=1,          # tune to your backend's limits
    optimization_level=3,
    mitigation_level=0,
)

kernel_model = HardwareKernelModel(circuit=circuit, runner=runner)

qiskit_runtime_service.__init__:WARNING:2026-04-06 23:52:18,528: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-04-06 23:52:19,206: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-04-06 23:52:22,806: Using instance: open-instance, plan: open


Using backend: ibm_kingston


In [20]:
"""
CentroidBasedKTA
================
Self-contained, single-class implementation.
- Pure NumPy (no JAX) — compatible with Qiskit kernels.
- **One** kernel_model.forward call per epoch covers all KAO + CO circuits.
- Drop this file into your notebook and instantiate directly.

Expected kernel_model interface
---------------------------------
    kernel_model.forward(x0, x1, weights)
        x0, x1  : np.ndarray  shape (M, D)
        weights  : np.ndarray  shape (M, *param_shape)  — per-pair weight rows
        returns  : np.ndarray  shape (M,)               — k(x0[i], x1[i])

    kernel_model.circuit.init_weights()
        returns  : np.ndarray  initial weight vector

    kernel_model.circuit_executions : int  (incremented by forward)

Hardware jobs per epoch
-----------------------
  Step                     This implementation
  ─────────────────────────────────────────────
  KAO + CO (fused)         1
  Alignment eval           1
  SVM train kernel         1
  SVM test kernel          1
  ─────────────────────────────────────────────
  Total                    4

  (original was  3·n_params + 3·D·(1 + n_sub) + 3  per epoch)

Note on KAO/CO ordering
-----------------------
Because KAO and CO circuits are batched together, both use the
*pre-epoch* weights for their kernel evaluations.  The KAO weight
update is applied first (in memory), and the CO centroid update is
then applied using the kernel values that were computed before that
weight change.  This is a single-step-lagged approximation that
costs nothing in hardware jobs.
"""

from __future__ import annotations

import time
from typing import Any

import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.svm import SVC
from tqdm import tqdm


class CentroidBasedKTA:
    """
    Alternating centroid-kernel optimisation (CCKA) for quantum kernels.

    Each epoch fuses all KAO and CO probe circuits into a single
    kernel_model.forward call, minimising hardware job submissions.

    Parameters
    ----------
    kernel_model    : object
        Must expose .forward(x0, x1, weights), .circuit.init_weights(),
        and .circuit_executions.
    data            : np.ndarray  shape (N, D)
    labels          : np.ndarray  shape (N,)
    split_size      : float       train fraction (default 0.8)
    matrix_type     : 'regular' | 'nystrom'
    landmark_points : int         Nystrom landmarks (required if nystrom)
    centering       : bool        apply kernel centering H K H
    epochs          : int
    centroids       : int         sub-centroids per class (default 4)
    clustering      : 'regular' | 'kmeans'
    lambda_co       : float       box-constraint regulariser weight
    lambda_kao      : float       L2 regulariser weight for KAO
    eps             : float       finite-difference step size
    alpha           : float       Newton step damping factor
    """

    # ══════════════════════════════════════════════════════════════════════
    # Construction
    # ══════════════════════════════════════════════════════════════════════

    def __init__(
        self,
        kernel_model,
        data: np.ndarray,
        labels: np.ndarray,
        *,
        split_size: float    = 0.8,
        matrix_type: str     = "regular",
        landmark_points: int = 0,
        centering: bool      = False,
        epochs: int          = 100,
        centroids: int       = 4,
        clustering: str      = "regular",
        lambda_co: float     = 0.01,
        lambda_kao: float    = 0.001,
        eps: float           = 0.01,
        alpha: float         = 0.1,
    ) -> None:

        # ── Validation ────────────────────────────────────────────────────
        if matrix_type not in ("regular", "nystrom"):
            raise ValueError("matrix_type must be 'regular' or 'nystrom'")
        if not (0.0 < split_size < 1.0):
            raise ValueError("split_size must be in (0, 1)")
        if matrix_type == "nystrom" and landmark_points <= 0:
            raise ValueError("landmark_points must be > 0 when matrix_type='nystrom'")

        # ── Hyperparameters ───────────────────────────────────────────────
        self.kernel_model    = kernel_model
        self.matrix_type     = matrix_type
        self.landmark_points = landmark_points
        self.centering       = centering
        self.epochs          = epochs
        self.n_centroids     = centroids
        self.use_kmeans      = clustering.lower() == "kmeans"
        self.lambda_co       = lambda_co
        self.lambda_kao      = lambda_kao
        self.eps             = eps
        self.alpha           = alpha

        # ── Data split ────────────────────────────────────────────────────
        data   = np.asarray(data,   dtype=np.float32)
        labels = np.asarray(labels, dtype=np.float32)
        self.xtrain, self.xtest, self.ytrain, self.ytest = self._split(
            data, labels, split_size, seed=42
        )

        # ── Feature range (for box constraint) ────────────────────────────
        self._feat_min = self.xtrain.min(axis=0)
        self._feat_max = self.xtrain.max(axis=0)

        # ── Weights ───────────────────────────────────────────────────────
        self.weights = np.asarray(
            kernel_model.circuit.init_weights(), dtype=np.float64
        )

        # ── Centroids ─────────────────────────────────────────────────────
        (
            self.main_centroids,
            self.main_centroid_labels,
            self.sub_centroids,
            self.sub_centroid_labels,
        ) = self._init_centroids()

    # ══════════════════════════════════════════════════════════════════════
    # Data helpers
    # ══════════════════════════════════════════════════════════════════════

    @staticmethod
    def _split(data, labels, split_size, seed=42):
        rng  = np.random.default_rng(seed)
        perm = rng.permutation(len(data))
        sp   = int(len(data) * split_size)
        tr, te = perm[:sp], perm[sp:]
        return data[tr], data[te], labels[tr], labels[te]

    # ══════════════════════════════════════════════════════════════════════
    # Centroid initialisation
    # ══════════════════════════════════════════════════════════════════════

    def _init_centroids(self):
        X, y          = self.xtrain, self.ytrain
        unique_labels = np.unique(y)
        n_cls, D      = len(unique_labels), X.shape[1]

        main_cents  = np.zeros((n_cls, D),                    dtype=np.float32)
        main_labels = np.zeros((n_cls,),                      dtype=np.float32)
        sub_cents   = np.zeros((n_cls * self.n_centroids, D), dtype=np.float32)
        sub_labels  = np.zeros((n_cls * self.n_centroids,),   dtype=np.float32)

        for ci, label in enumerate(unique_labels):
            cls_data = X[y == label]
            main_cents[ci]  = cls_data.mean(axis=0)
            main_labels[ci] = float(label)

            if self.use_kmeans and len(cls_data) >= self.n_centroids:
                km = KMeans(
                    n_clusters=self.n_centroids, n_init="auto", random_state=42
                ).fit(cls_data)
                sc = km.cluster_centers_.astype(np.float32)
            else:
                chunks = np.array_split(cls_data, self.n_centroids)
                sc = np.stack([c.mean(axis=0) for c in chunks]).astype(np.float32)

            for si in range(self.n_centroids):
                idx = ci * self.n_centroids + si
                sub_cents[idx]  = sc[si]
                sub_labels[idx] = float(label)

        return main_cents, main_labels, sub_cents, sub_labels

    # ══════════════════════════════════════════════════════════════════════
    # Kernel matrix helpers  (used for alignment + SVM only)
    # ══════════════════════════════════════════════════════════════════════

    def _kernel_matrix(self, weights, X):
        if self.matrix_type == "regular":
            return self._regular_km(weights, X)
        return self._nystrom_km(weights, X)

    def _regular_km(self, weights, X):
        N      = len(X)
        iu, ju = np.triu_indices(N)
        # broadcast single weight vector to all pairs
        w_rep  = np.repeat(weights[np.newaxis, ...], len(iu), axis=0)
        k_vals = np.asarray(self.kernel_model.forward(X[iu], X[ju], w_rep))
        K      = np.zeros((N, N), dtype=np.float64)
        K[iu, ju] = k_vals
        K[ju, iu] = k_vals
        return K

    def _nystrom_km(self, weights, X):
        M, N = self.landmark_points, len(X)
        lm   = X[:M]
        ri   = np.repeat(np.arange(N), M)
        ci   = np.tile(np.arange(M), N)
        w_rep = np.repeat(weights[np.newaxis, ...], len(ri), axis=0)
        KNM  = np.asarray(
            self.kernel_model.forward(X[ri], lm[ci], w_rep)
        ).reshape(N, M)
        ri2, ci2 = np.repeat(np.arange(M), M), np.tile(np.arange(M), M)
        w_rep2   = np.repeat(weights[np.newaxis, ...], M * M, axis=0)
        KMM      = np.asarray(
            self.kernel_model.forward(lm[ri2], lm[ci2], w_rep2)
        ).reshape(M, M)
        KMM_inv = np.linalg.inv(KMM + 1e-8 * np.eye(M))
        return KNM @ KMM_inv @ KNM.T

    def _test_km(self, weights, X_train, X_test):
        N, M  = len(X_train), len(X_test)
        x0    = np.repeat(X_test, N, axis=0)
        x1    = np.tile(X_train, (M, 1))
        w_rep = np.repeat(weights[np.newaxis, ...], N * M, axis=0)
        return np.asarray(
            self.kernel_model.forward(x0, x1, w_rep)
        ).reshape(M, N)

    def _center(self, K):
        if not self.centering:
            return K
        n = K.shape[0]
        H = np.eye(n) - np.ones((n, n)) / n
        return H @ K @ H

    # ══════════════════════════════════════════════════════════════════════
    # KTA / loss helpers
    # ══════════════════════════════════════════════════════════════════════

    def alignment(self, weights, X, y):
        K    = self._center(self._kernel_matrix(weights, X))
        T    = np.outer(y, y)
        norm = np.linalg.norm(K, "fro") * np.linalg.norm(T, "fro")
        return float(np.sum(K * T) / (norm + 1e-10))

    @staticmethod
    def _kta_vec(K_vec, Y, l):
        """KTA from a kernel vector K_vec = [k(main_c, sub_c_i)]."""
        num = l * np.dot(K_vec, Y)
        den = np.linalg.norm(K_vec) * np.linalg.norm(Y)
        return num / (den + 1e-10)

    def _l2(self, weights):
        flat = weights.ravel()
        return float(np.dot(flat, flat) / max(len(flat), 1))

    def _box_penalty(self, centroid):
        return float(
            np.sum(
                np.maximum(centroid - self._feat_max, 0.0)
                + np.maximum(self._feat_min - centroid, 0.0)
            )
        )

    # ══════════════════════════════════════════════════════════════════════
    # SVM evaluation
    # ══════════════════════════════════════════════════════════════════════

    def svm_training(self, X, y):
        K_train = self._center(self._kernel_matrix(self.weights, X))
        svm     = SVC(kernel="precomputed", C=1.0, probability=True, max_iter=10_000)
        svm.fit(K_train, y)

        K_test_raw = self._test_km(self.weights, self.xtrain, self.xtest)
        if self.centering:
            K_test = (
                K_test_raw
                - K_test_raw.mean(axis=1, keepdims=True)
                - K_train.mean(axis=0, keepdims=True)
                + K_train.mean()
            )
        else:
            K_test = K_test_raw

        y_pred_tr = svm.predict(K_train)
        y_pred_te = svm.predict(K_test)

        return {
            "svm":             svm,
            "train_accuracy":  float(accuracy_score(y,          y_pred_tr)),
            "test_accuracy":   float(accuracy_score(self.ytest,  y_pred_te)),
            "f1_score":        float(f1_score(       self.ytest,  y_pred_te, average="macro")),
            "precision_score": float(precision_score(self.ytest,  y_pred_te, average="macro")),
            "recall_score":    float(recall_score(   self.ytest,  y_pred_te, average="macro")),
        }

    # ══════════════════════════════════════════════════════════════════════
    # FUSED EPOCH STEP — one forward call for all KAO + CO circuits
    #
    # Batch layout (total M pairs in the single forward call):
    #
    #   ┌─────────────────────────────────────────────────────────────────┐
    #   │  KAO section      3 × n_params × N_sub  pairs                  │
    #   │    Each of the 3·n_params shifted weight vectors is paired      │
    #   │    with the N_sub sub-centroids (x0 = main_centroid).           │
    #   ├─────────────────────────────────────────────────────────────────┤
    #   │  CO main-cent     D × 3 × N_sub  pairs                         │
    #   │    main_centroid perturbed by ±eps / 0 per feature dim,         │
    #   │    paired with the full X_cl; weights = pre-epoch self.weights. │
    #   ├─────────────────────────────────────────────────────────────────┤
    #   │  CO sub-cents     n_sub × D × 3 × N_sub  pairs                 │
    #   │    Each sub-centroid (row in X_cl) perturbed by ±eps / 0 per    │
    #   │    feature dim; x0 = main_centroid; weights = pre-epoch.        │
    #   └─────────────────────────────────────────────────────────────────┘
    #
    # After the single forward call:
    #   1. KAO arctan2 rule  → update self.weights
    #   2. CO Newton step    → update self.main_centroids, self.sub_centroids
    #      (using kernel values computed with pre-epoch weights — single-step lag)
    #
    # Returns
    # -------
    # loss_log : float   KAO loss at the pre-epoch weights (no extra job)
    # ══════════════════════════════════════════════════════════════════════

    def _epoch_step_batched(
        self,
        main_centroid: np.ndarray,
        X_cl: np.ndarray,
        Y_cl: np.ndarray,
        l_kao: float,
        l_co: float,
        cl_kao: float,
        mask_kao: np.ndarray,
    ) -> float:

        flat        = self.weights.ravel().copy()          # pre-epoch flat weights
        n_params    = len(flat)
        param_shape = self.weights.shape
        N_sub       = len(X_cl)
        D           = self.main_centroids.shape[1]
        main_idx    = int(np.where(self.main_centroid_labels == cl_kao)[0][0])
        sub_indices = np.where(mask_kao)[0]
        n_sub       = len(sub_indices)
        mc          = self.main_centroids[main_idx].copy()  # snapshot for CO

        # ── Convenience: broadcast a single weight to M pairs ─────────────
        def _rep_w(w, M):
            """Repeat weight vector w along a new leading axis M times."""
            return np.repeat(w[np.newaxis, ...], M, axis=0)

        # ══════════════════════════════════════════════════════════════════
        # SECTION 1 — KAO probe circuits
        #   3 offsets × n_params shifted weight vectors × N_sub pairs
        #   Total: 3 * n_params * N_sub
        # ══════════════════════════════════════════════════════════════════
        offsets   = np.array([0.0, np.pi / 2, np.pi])            # (3,)
        all_flats = np.tile(flat, (3, n_params, 1))               # (3, P, P)
        for o, off in enumerate(offsets):
            all_flats[o, np.arange(n_params), np.arange(n_params)] += off
        all_flats_2d = all_flats.reshape(3 * n_params, n_params)  # (3P, P)
        all_weights  = all_flats_2d.reshape(3 * n_params, *param_shape)  # (3P, *shape)

        # x0 = main_centroid (N_sub copies), tiled for all 3P weight variants
        x0_base = np.repeat(main_centroid[np.newaxis, :], N_sub, axis=0)  # (N_sub, D)
        x0_kao  = np.tile(x0_base, (3 * n_params, 1))                      # (3P*N_sub, D)
        x1_kao  = np.tile(X_cl,    (3 * n_params, 1))                      # (3P*N_sub, D)
        w_kao   = np.repeat(all_weights, N_sub, axis=0)                    # (3P*N_sub, *shape)

        kao_size = 3 * n_params * N_sub

        # ══════════════════════════════════════════════════════════════════
        # SECTION 2 — CO main-centroid probe circuits
        #   D dims × 3 offsets × N_sub pairs; weights = pre-epoch
        #   Total: D * 3 * N_sub
        # ══════════════════════════════════════════════════════════════════
        co_main_x0_list: list[np.ndarray] = []
        co_main_x1_list: list[np.ndarray] = []

        for d in range(D):
            for off in (-self.eps, 0.0, self.eps):
                mc_p = mc.copy()
                mc_p[d] += off
                co_main_x0_list.append(np.repeat(mc_p[np.newaxis, :], N_sub, axis=0))
                co_main_x1_list.append(X_cl.copy())

        co_main_size = D * 3 * N_sub

        if co_main_x0_list:
            x0_co_main = np.concatenate(co_main_x0_list, axis=0)   # (D*3*N_sub, D)
            x1_co_main = np.concatenate(co_main_x1_list, axis=0)
            w_co_main  = _rep_w(self.weights, co_main_size)          # (D*3*N_sub, *shape)
        else:
            x0_co_main = np.empty((0, D), dtype=np.float32)
            x1_co_main = np.empty((0, D), dtype=np.float32)
            w_co_main  = np.empty((0, *param_shape), dtype=np.float64)

        # ══════════════════════════════════════════════════════════════════
        # SECTION 3 — CO sub-centroid probe circuits
        #   n_sub × D dims × 3 offsets × N_sub pairs; weights = pre-epoch
        #   x0 = mc fixed, x1 = X_cl with sub-centroid row perturbed
        #   Total: n_sub * D * 3 * N_sub
        # ══════════════════════════════════════════════════════════════════
        co_sub_x0_list: list[np.ndarray] = []
        co_sub_x1_list: list[np.ndarray] = []

        for i, si in enumerate(sub_indices):
            # local_row: row of X_cl that belongs to sub-centroid si
            local_row = int(np.searchsorted(sub_indices, si))
            for d in range(D):
                for off in (-self.eps, 0.0, self.eps):
                    X_cl_p = X_cl.copy()
                    X_cl_p[local_row, d] += off
                    co_sub_x0_list.append(np.repeat(mc[np.newaxis, :], N_sub, axis=0))
                    co_sub_x1_list.append(X_cl_p)

        co_sub_size = n_sub * D * 3 * N_sub

        if co_sub_x0_list:
            x0_co_sub = np.concatenate(co_sub_x0_list, axis=0)      # (n_sub*D*3*N_sub, D)
            x1_co_sub = np.concatenate(co_sub_x1_list, axis=0)
            w_co_sub  = _rep_w(self.weights, co_sub_size)             # (n_sub*D*3*N_sub, *shape)
        else:
            x0_co_sub = np.empty((0, D), dtype=np.float32)
            x1_co_sub = np.empty((0, D), dtype=np.float32)
            w_co_sub  = np.empty((0, *param_shape), dtype=np.float64)

        # ══════════════════════════════════════════════════════════════════
        # ── SINGLE FORWARD CALL ───────────────────────────────────────────
        # ══════════════════════════════════════════════════════════════════
        x0_all = np.concatenate([x0_kao,   x0_co_main, x0_co_sub], axis=0)
        x1_all = np.concatenate([x1_kao,   x1_co_main, x1_co_sub], axis=0)
        w_all  = np.concatenate([w_kao,    w_co_main,  w_co_sub],  axis=0)

        k_flat = np.asarray(
            self.kernel_model.forward(x0_all, x1_all, w_all)
        )   # shape (kao_size + co_main_size + co_sub_size,)

        # Slice into the three sections
        k_kao_flat     = k_flat[:kao_size]
        k_co_main_flat = k_flat[kao_size : kao_size + co_main_size]
        k_co_sub_flat  = k_flat[kao_size + co_main_size :]

        # ══════════════════════════════════════════════════════════════════
        # APPLY KAO UPDATE (arctan2 rule, vectorised over all params)
        # ══════════════════════════════════════════════════════════════════
        k_vals = k_kao_flat.reshape(3, n_params, N_sub)    # (3, P, N_sub)

        # KTA per (offset, param)
        dots   = np.einsum("opn,n->op", k_vals, Y_cl)      # (3, P)
        norms  = np.linalg.norm(k_vals, axis=2)            # (3, P)
        y_norm = np.linalg.norm(Y_cl)
        kta    = (l_kao * dots) / (norms * y_norm + 1e-10) # (3, P)

        # L2 per shifted weight set (CPU only)
        l2   = (np.sum(all_flats_2d ** 2, axis=1) / max(n_params, 1)
                ).reshape(3, n_params)

        loss_matrix = 1.0 - kta + self.lambda_kao * l2     # (3, P)

        L0, L_pi2, L_pi = loss_matrix[0], loss_matrix[1], loss_matrix[2]
        numer  = 2.0 * L_pi2 - L_pi - L0
        denom  = L_pi - L0
        deltas = np.arctan2(numer, denom + 1e-10)           # (P,)
        self.weights = (flat - deltas).reshape(param_shape)

        # Loss log — reuse k_vals[0, 0, :] = kernel(mc, X_cl) at original weights
        # (offset=0 applied to param 0 leaves all weights unchanged)
        k_baseline = k_vals[0, 0, :]
        kta_log    = self._kta_vec(k_baseline, Y_cl, l_kao)
        loss_log   = float(1.0 - kta_log + self.lambda_kao * self._l2(self.weights))

        # ══════════════════════════════════════════════════════════════════
        # APPLY CO MAIN-CENTROID UPDATE (finite-difference Newton)
        # ══════════════════════════════════════════════════════════════════
        # k_co_main reshaped to (D, 3, N_sub); axis-1 order: [-eps, 0, +eps]
        k_co_main = k_co_main_flat.reshape(D, 3, N_sub)

        mc_cur = self.main_centroids[main_idx]   # live view for the closure

        for d in range(D):
            k_minus = k_co_main[d, 0]
            k_zero  = k_co_main[d, 1]
            k_plus  = k_co_main[d, 2]
            cur_val = mc_cur[d]

            def _loss_main(k_vec, probe_val, _d=d):
                kta_   = self._kta_vec(k_vec, Y_cl, l_co)
                mc_p   = mc_cur.copy()
                mc_p[_d] = probe_val
                return 1.0 - kta_ + self.lambda_co * self._box_penalty(mc_p)

            L_m = _loss_main(k_minus, cur_val - self.eps)
            L0_ = _loss_main(k_zero,  cur_val)
            L_p = _loss_main(k_plus,  cur_val + self.eps)

            grad = (L_p - L_m) / (2.0 * self.eps)
            hess = (L_p - 2.0 * L0_ + L_m) / (self.eps ** 2 + 1e-10)

            if abs(hess) < 1e-6 or np.isnan(hess) or np.isnan(grad):
                step = -self.alpha * grad
            else:
                step = -self.alpha * grad / (abs(hess) + 1e-10)

            new_val = np.clip(cur_val + step, self._feat_min[d], self._feat_max[d])
            cand_k  = k_plus if step > 0 else k_minus
            if _loss_main(cand_k, new_val) < L0_:
                self.main_centroids[main_idx, d] = new_val
            # mc_cur is a view of self.main_centroids[main_idx], so dim d+1
            # onward sees the just-updated value — same behaviour as original.

        # ══════════════════════════════════════════════════════════════════
        # APPLY CO SUB-CENTROID UPDATES (finite-difference Newton)
        # ══════════════════════════════════════════════════════════════════
        # k_co_sub_flat layout: sub_i block, then dim d block, then 3 offsets,
        # each block has N_sub values.
        block_per_sub = D * 3 * N_sub   # flat bytes per sub-centroid

        for i, si in enumerate(sub_indices):
            k_si   = k_co_sub_flat[i * block_per_sub : (i + 1) * block_per_sub
                                   ].reshape(D, 3, N_sub)
            sc_cur = self.sub_centroids[si]   # live view

            for d in range(D):
                k_minus = k_si[d, 0]
                k_zero  = k_si[d, 1]
                k_plus  = k_si[d, 2]
                cur_val = sc_cur[d]

                def _loss_sub(k_vec):
                    return 1.0 - self._kta_vec(k_vec, Y_cl, l_co)

                L_m = _loss_sub(k_minus)
                L0_ = _loss_sub(k_zero)
                L_p = _loss_sub(k_plus)

                grad = (L_p - L_m) / (2.0 * self.eps)
                hess = (L_p - 2.0 * L0_ + L_m) / (self.eps ** 2 + 1e-10)

                if abs(hess) < 1e-6 or np.isnan(hess) or np.isnan(grad):
                    step = self.alpha * grad
                else:
                    step = self.alpha * grad / (abs(hess) + 1e-10)

                new_val = np.clip(cur_val + step, self._feat_min[d], self._feat_max[d])
                cand_k  = k_plus if step > 0 else k_minus
                if _loss_sub(cand_k) < L0_:
                    self.sub_centroids[si, d] = new_val
                    sc_cur = self.sub_centroids[si]   # refresh view after update

        return loss_log

    # ══════════════════════════════════════════════════════════════════════
    # MAIN TRAINING LOOP
    # ══════════════════════════════════════════════════════════════════════

    def align(self) -> dict[str, Any]:
        """
        Run the alternating KAO / CO optimisation and return history.

        Hardware jobs per epoch
        -----------------------
          Step                     Jobs
          ──────────────────────────────────────
          KAO + CO (fused)         1
          Alignment eval           1
          SVM train kernel         1
          SVM test  kernel         1
          ──────────────────────────────────────
          Total                    4
          (original: 3·n_params + 3·D·(1+n_sub) + 3)

        Returns
        -------
        dict with keys:
            weights, main_centroids, sub_centroids,
            init_train_accuracy, init_test_accuracy,
            alignment_history, loss_history,
            train_accuracy_history, test_accuracy_history,
            best_test_accuracy, final_svm_metrics,
            f1_score_history, precision_score_history, recall_score_history,
            time, circuit_executions
        """
        init = self.svm_training(self.xtrain, self.ytrain)

        alignment_hist   = []
        loss_hist        = []
        train_acc        = []
        test_acc         = []
        f1s, precs, recs = [], [], []
        main_cent_hist   = []
        sub_cent_hist    = []

        unique_labels = np.unique(self.ytrain)
        n_cls         = len(unique_labels)

        best_test_acc       = -np.inf
        best_weights        = self.weights.copy()
        best_main_centroids = self.main_centroids.copy()
        best_sub_centroids  = self.sub_centroids.copy()

        start = time.perf_counter()

        for epoch in tqdm(range(self.epochs), desc="[CentroidBasedKTA] KTA alignment"):

            # ── Select class for this epoch ────────────────────────────────
            cl_kao = unique_labels[epoch % n_cls]
            l_kao  =  float(cl_kao)   # KAO: +current_label
            l_co   = -float(cl_kao)   # CO:  -current_label

            mask_kao      = self.sub_centroid_labels == cl_kao
            X_cl          = self.sub_centroids[mask_kao].copy()
            Y_cl          = np.where(
                self.sub_centroid_labels[mask_kao] == cl_kao, 1.0, -1.0
            ).astype(np.float64)

            main_idx      = int(np.where(self.main_centroid_labels == cl_kao)[0][0])
            main_centroid = self.main_centroids[main_idx].copy()

            # ── FUSED KAO + CO: one hardware job ──────────────────────────
            loss_val = self._epoch_step_batched(
                main_centroid, X_cl, Y_cl,
                l_kao=l_kao, l_co=l_co,
                cl_kao=cl_kao, mask_kao=mask_kao,
            )
            loss_hist.append(loss_val)

            main_cent_hist.append(self.main_centroids.copy())
            sub_cent_hist.append(self.sub_centroids.copy())

            # ── Alignment (one kernel matrix call) ─────────────────────────
            alignment_hist.append(
                self.alignment(self.weights, self.xtrain, self.ytrain)
            )

            # ── SVM evaluation (two kernel matrix calls) ───────────────────
            result = self.svm_training(self.xtrain, self.ytrain)
            train_acc.append(result["train_accuracy"])
            test_acc.append(result["test_accuracy"])
            f1s.append(result["f1_score"])
            precs.append(result["precision_score"])
            recs.append(result["recall_score"])

            if result["test_accuracy"] > best_test_acc:
                best_test_acc       = result["test_accuracy"]
                best_weights        = self.weights.copy()
                best_main_centroids = self.main_centroids.copy()
                best_sub_centroids  = self.sub_centroids.copy()

        # ── Restore best checkpoint ────────────────────────────────────────
        self.weights        = best_weights
        self.main_centroids = best_main_centroids
        self.sub_centroids  = best_sub_centroids

        final_result = self.svm_training(self.xtrain, self.ytrain)
        train_acc.append(final_result["train_accuracy"])
        test_acc.append(final_result["test_accuracy"])
        f1s.append(final_result["f1_score"])
        precs.append(final_result["precision_score"])
        recs.append(final_result["recall_score"])

        return {
            "weights":                 self.weights,
            "main_centroids":          main_cent_hist,
            "sub_centroids":           sub_cent_hist,
            "init_train_accuracy":     init["train_accuracy"],
            "init_test_accuracy":      init["test_accuracy"],
            "alignment_history":       alignment_hist,
            "loss_history":            loss_hist,
            "train_accuracy_history":  train_acc,
            "test_accuracy_history":   test_acc,
            "best_test_accuracy":      float(best_test_acc),
            "final_svm_metrics":       final_result,
            "f1_score_history":        f1s,
            "precision_score_history": precs,
            "recall_score_history":    recs,
            "time":                    time.perf_counter() - start,
            "circuit_executions":      self.kernel_model.circuit_executions,
        }

In [21]:
# ── Centroid-Based KTA (recommended for hardware — fewest kernel evaluations) ────
aligner = CentroidBasedKTA(
                    kernel_model= kernel_model,
                    data = X,
                    labels = y,
                    matrix_type='regular',
                    clustering='regular',
                    split_size=0.50,
                    centroids= 2,
                    lambda_co=0.0,
                    lambda_kao=0.0,
                    epochs=1,
                    eps=0.001,
                    alpha=0.01
        )

history = aligner.align()
optimized_weights = history["weights"]
print(f"Best test accuracy : {history['best_test_accuracy']:.4f}")
print(f"Circuit executions : {history['circuit_executions']}")

Submitting batches:   0%|          | 0/10 [00:00<?, ?it/s]


TypeError: only length-1 arrays can be converted to Python scalars